# Проект автоматизации BI-аналитики Wildberries
## Модуль: Проверка инфраструктуры и подключения к DWH

**Бизнес-цель:** Проверить корректность конфигурации окружения, доступность локальной базы данных PostgreSQL (слой DWH) и безопасность хранения учетных данных.

**Используемые инструменты:** 
* Python (библиотеки `os`, `psycopg2`, `python-dotenv`)
* PostgreSQL (целевая СУБД)


In [ ]:
import os
from dotenv import load_dotenv

# Определяем путь к корню проекта и файлу .env, который лежит в папке scripts
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, 'scripts', '.env')

print(f"Ищем файл .env по пути: {env_path}")

# Загружаем переменные из .env
if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path)
    
    # Проверяем наличие ключевых переменных (без вывода их значений на экран!)
    db_name = os.getenv("DB_NAME")
    db_user = os.getenv("DB_USER")
    
    if db_name and db_user:
        print("✅ Успешно: Переменные окружения загружены. База данных и пользователь распознаны.")
    else:
        print("❌ Ошибка: Файл .env найден, но внутри отсутствуют переменные DB_NAME или DB_USER.")
else:
    print("❌ Критическая ошибка: Файл .env не найден по указанному пути. Проверьте структуру папок.")


**Результат выполнения проверки окружения:**
Ищем файл .env по пути: d:\Documents\Dev\projects\new-bi-project\scripts\.env
✅ Успешно: Переменные окружения загружены. База данных и пользователь распознаны.

In [ ]:
import psycopg2

try:
    # Инициализируем подключение, используя переменные из окружения
    connection = psycopg2.connect(
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5432")
    )
    
    # Создаем курсор для выполнения запросов
    cursor = connection.cursor()
    
    # Выполняем проверочный запрос к системной функции PostgreSQL
    cursor.execute("SELECT version();")
    db_version = cursor.fetchone()
    
    print("✅ Успешно: Физическое подключение к PostgreSQL установлено.")
    print(f"Версия СУБД в вашей системе:\n{db_version[0]}")
    
except Exception as e:
    print(f"❌ Ошибка при подключении к базе данных: {e}")
    print("Рекомендация: Проверьте, запущен ли сервер PostgreSQL и верны ли доступы в .env.")
    
finally:
    # Обязательно закрываем ресурсы, это признак хорошего тона в разработке
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'connection' in locals() and connection:
        connection.close()
        print("Сессия подключения закрыта.")


**Результат проверки подключения к СУБД:**

✅ Успешно: Физическое подключение к PostgreSQL установлено.
Версия СУБД в вашей системе:
PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit
Сессия подключения закрыта.

### Шаг 4: Проверка успешного создания структуры в схеме staging

Проверим физическое наличие созданной таблицы `raw_financial_reports` в базе данных и выведем её структуру (названия колонок), чтобы убедиться, что типы данных и схема инициализированы корректно.


In [ ]:
try:
    connection = psycopg2.connect(
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5432")
    )
    cursor = connection.cursor()
    
    # Запрос к системному каталогу информации для получения колонок таблицы
    query = """
        SELECT column_name, data_type 
        FROM information_schema.columns 
        WHERE table_schema = 'staging' AND table_name = 'raw_financial_reports'
        ORDER BY ordinal_position;
    """
    cursor.execute(query)
    columns = cursor.fetchall()
    
    print(f"✅ Успешно: Таблица найдена. Всего колонок создано: {len(columns)}\n")
    print("Первые 5 колонок для демонстрации структуры:")
    for col in columns[:5]:
        print(f"Поле: {col[0]:<25} | Тип данных: {col[1]}")
        
except Exception as e:
    print(f"❌ Ошибка при проверке таблицы: {e}")
finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'connection' in locals() and connection:
        connection.close()


**Результат проверки структуры таблицы:**

✅ Успешно: Таблица найдена. Всего колонок создано: 42

Первые 5 колонок для демонстрации структуры:
Поле: raw_id                    | Тип данных: bigint
Поле: created_at                | Тип данных: timestamp with time zone
Поле: processed_at              | Тип данных: timestamp with time zone
Поле: realizationreport_id      | Тип данных: bigint
Поле: srv_distrib_date          | Тип данных: timestamp with time zone